# VR Disaster Management AI — ML Model Training Notebook

This notebook trains an **XGBoost / Random Forest Regression Model** to evaluate trainee VR disaster simulation telemetry.

### Pipeline Overview:
1. **Synthetic Telemetry Dataset Generation** (1,500 simulation sessions)
2. **Feature Engineering** (Reaction time, hazard avoidance, protocol safety compliance, casualty rescue efficiency)
3. **Model Training & Hyperparameter Fitting**
4. **Model Performance Evaluation** ($R^2$ Score, MSE, Feature Importances)
5. **Model Export** to `xgboost_model.pkl` for FastAPI inference.

In [1]:
import numpy as np
import pandas as pd
import pickle
import os
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

## 1. Dataset Generation

In [2]:
def generate_dataset(n_samples=1500, random_state=42):
    np.random.seed(random_state)
    evacuation_time = np.random.uniform(60, 600, n_samples)
    avg_reaction_time = np.random.uniform(200, 1500, n_samples)
    correct_decisions = np.random.randint(2, 12, n_samples)
    wrong_decisions = np.random.randint(0, 5, n_samples)
    safety_violations = np.random.randint(0, 4, n_samples)
    hazards_detected = np.random.randint(1, 10, n_samples)
    victims_rescued = np.random.randint(0, 6, n_samples)
    objectives_completed = np.random.randint(1, 5, n_samples)

    decision_ratio = correct_decisions / (correct_decisions + wrong_decisions + 1e-5)
    speed_factor = np.clip(1.0 - (avg_reaction_time / 2000.0), 0.1, 1.0)
    safety_factor = np.clip(1.0 - (safety_violations * 0.25), 0.0, 1.0)
    rescue_factor = np.clip(victims_rescued / 5.0, 0.0, 1.0)

    score = (decision_ratio * 35) + (safety_factor * 25) + (rescue_factor * 25) + (speed_factor * 15)
    score = np.clip(score + np.random.normal(0, 2.5, n_samples), 0, 100)

    return pd.DataFrame({
        'evacuation_time': evacuation_time,
        'avg_reaction_time': avg_reaction_time,
        'correct_decisions': correct_decisions,
        'wrong_decisions': wrong_decisions,
        'safety_violations': safety_violations,
        'hazards_detected': hazards_detected,
        'victims_rescued': victims_rescued,
        'objectives_completed': objectives_completed,
        'performance_score': score
    })

df = generate_dataset()
print(f"Generated dataset shape: {df.shape}")
df.head()

## 2. Model Training & Evaluation

In [3]:
X = df.drop(columns=['performance_score'])
y = df['performance_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"R^2 Score: {r2_score(y_test, y_pred):.4f}")
print(f"Mean Squared Error (MSE): {mean_squared_error(y_test, y_pred):.4f}")

## 3. Model Export

In [4]:
os.makedirs('../models', exist_ok=True)
with open('../models/xgboost_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print('Successfully exported model pickle to ../models/xgboost_model.pkl')